### Preventing Race Conditions with Mutex Locks (threading.Lock())

In [39]:
import threading
import time

class BankAccount:
    def __init__(self, user: str, balance: float) -> None:
        self.__user = user
        self.__balance = balance
        self.__lock = threading.Lock()

    @property
    def balance(self):
        return self.__balance

    def withdraw(self, amount):
        with self.__lock:
            if amount > self.__balance:
                raise ValueError(f"Insufficient funds.")
            self.__balance -= amount
            print(f"Withdrawal: {amount}, Remaining Balance: {self.__balance}")

bank_account = BankAccount("Vishal", 100)
threads = [threading.Thread(target=bank_account.withdraw, args=(num,)) for num in [10, 50, 10, 10, 10, 10]]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(f"Final account balance is {bank_account.balance}")

Withdrawal: 10, Remaining Balance: 90
Withdrawal: 10, Remaining Balance: 80
Withdrawal: 10, Remaining Balance: 70
Withdrawal: 50, Remaining Balance: 20
Withdrawal: 10, Remaining Balance: 10
Withdrawal: 10, Remaining Balance: 0
Final account balance is 0


###  RLock

In [72]:
import threading

# If we would have used generic lock then reacquiring the lock 
# will cause deadlock. But with RLock, we can reacquire it.
rlock = threading.RLock()

def task(n):
    print(f"{n} trying to acquire lock...")
    with rlock:
        print(f"Lock acquired by {n}, working...")
        with rlock:
            print(f"Lock re-acquired safely by {n}!")

threads = [threading.Thread(target=task, args=(i,)) for i in range(2)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

0 trying to acquire lock...
Lock acquired by 0, working...
Lock re-acquired safely by 0!
1 trying to acquire lock...
Lock acquired by 1, working...
Lock re-acquired safely by 1!


### Semaphore

In [4]:
import threading
import time

atm_count = 3
atm_semaphore = threading.Semaphore(3)

def task(n):
    global atm_count
    print(f"Customer {n} is waiting outside the ATM.")
    with atm_semaphore:
        atm_count -= 1
        print(f"Customer {n} is using an ATM. Current: {atm_count}")
        time.sleep(n * 2)
        atm_count += 1
        print(f"Customer {n} has finished using the ATM.")

threads = [threading.Thread(target=task, args=(i,)) for i in range(5)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

Customer 0 is waiting outside the ATM.
Customer 0 is using an ATM. Current: 2
Customer 0 has finished using the ATM.
Customer 1 is waiting outside the ATM.
Customer 1 is using an ATM. Current: 2
Customer 2 is waiting outside the ATM.
Customer 2 is using an ATM. Current: 1
Customer 3 is waiting outside the ATM.
Customer 3 is using an ATM. Current: 0
Customer 4 is waiting outside the ATM.
Customer 1 has finished using the ATM.
Customer 4 is using an ATM. Current: 0
Customer 2 has finished using the ATM.
Customer 3 has finished using the ATM.
Customer 4 has finished using the ATM.
